In [44]:
import HierarchiaPy
import os
import h5py
import numpy as np
import pandas as pd
import scipy as sp
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, resample_poly
from scipy.signal import butter, filtfilt, resample_poly, resample, find_peaks
import neurokit2 as nk
from scipy.signal import find_peaks
import seaborn as sns
from scipy.stats import wilcoxon
import scipy

# Pandas display settings (optional)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 0)
print(HierarchiaPy.__version__)

0.2.6


In [ ]:
file = r"C:\Users\sjs93\Downloads\CD1_food_comp_all_data_model 1.xlsx"

xls = pd.ExcelFile(file)

print(xls.sheet_names)

['Food_competition_D1_FEMALE', 'Food_Competition_D2_FEMALE', 'Food_Comp_D1_MALE', 'Food_Comp_D2_MALE']


In [27]:
df = pd.read_excel(
    file,
    sheet_name="Food_competition_D1_FEMALE"
)

print(df.head())
print(df.columns)

     Day 2 Cage 1 Unnamed: 2 Unnamed: 3  Unnamed: 4 Unnamed: 5 Cage 2  \
0      NaN  match    Winners     Losers         NaN        NaN  match   
1  trial 1   2vs3        1.3        1.2         NaN    trial 1   2vs3   
2  trial 1   4vs1        1.1        1.4         NaN    trial 1   4vs1   
3  trial 1   3vs4        1.3        1.4         NaN    trial 1   3vs4   
4  trial 1   2vs1        1.1        1.2         NaN    trial 1   2vs1   

  Unnamed: 7 Unnamed: 8  Unnamed: 9 Unnamed: 10 Cage 3 Unnamed: 12  \
0    Winners     Losers         NaN         NaN  match     Winners   
1        2.3        2.2         NaN     trial 1   2vs3         3.3   
2        2.4        2.1         NaN     trial 1   4vs1         3.4   
3        2.4        2.3         NaN     trial 1   3vs4         3.3   
4        2.2        2.1         NaN     trial 1   2vs1         3.1   

  Unnamed: 13  Unnamed: 14 Unnamed: 15 Cage 4 Unnamed: 17 Unnamed: 18  \
0      Losers          NaN         NaN  match     Winners      Lose

In [30]:
print(dir(HierarchiaPy))

['Hierarchia', 'HierarchiaPy', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', 'methods', 'metrics', 'name', 'utilities']


In [31]:
help(HierarchiaPy)

Help on package HierarchiaPy:

NAME
    HierarchiaPy - # HierarchiaPy - Statistical tool to calculate dominance/hierarchy

PACKAGE CONTENTS
    HierarchiaPy

SUBMODULES
    methods
    metrics
    utilities

DATA
    name = 'HierarchiaPy'

VERSION
    0.2.6

FILE
    c:\users\sjs93\miniconda3\envs\biopipeline-env\lib\site-packages\hierarchiapy\__init__.py




In [32]:
# Path to Excel file containing food competition data
file = r"C:\Users\sjs93\Downloads\CD1_food_comp_all_data_model.xlsx"

# Load the first sheet (Day 2 Female)
df = pd.read_excel(file, sheet_name=0)

# Extract Cage 1 winner/loser columns
# Column 2 = Winners
# Column 3 = Losers
# Skip the first row because it contains headers inside the spreadsheet
cage1 = df.iloc[1:, [2,3]].copy()

# Rename columns to what HierarchiaPy expects
cage1.columns = ["winner", "loser"]

# Remove empty rows
cage1 = cage1.dropna()

# Create hierarchy object from winner-loser interactions
# Each row represents one contest:
# winner defeated loser
hier = Hierarchia(cage1, "winner", "loser")

# Print pairwise dominance matrix
print(hier.mat)

# Print animal IDs corresponding to matrix rows/columns
print(hier.indices)

[[0 1 3 2]
 [2 0 1 1]
 [0 2 0 2]
 [1 2 1 0]]
[1.1, 1.2, 1.3, 1.4]


In [12]:
print(cage1.head(20))

   winner loser
1     1.3   1.2
2     1.1   1.4
3     1.3   1.4
4     1.1   1.2
5     1.1   1.3
6     1.4   1.2
7     1.1   1.3
8     1.2   1.3
9     1.4   1.2
10    1.4   1.1
11    1.2   1.1
12    1.3   1.4
13    1.2   1.1
14    1.3   1.2
15    1.4   1.3
16    1.2   1.4
17    1.1   1.3
18    1.1   1.4


In [33]:
xls = pd.ExcelFile(file)

print(xls.sheet_names)

['Food_competition_D1_FEMALE', 'Food_Competition_D2_FEMALE', 'Food_Comp_D1_MALE', 'Food_Comp_D2_MALE']


In [34]:
for sheet in xls.sheet_names:
    df = pd.read_excel(file, sheet_name=sheet)

    print("\n" + "="*50)
    print(sheet)
    print("="*50)

    print(df.columns.tolist())
    print(df.head(3))


Food_competition_D1_FEMALE
['Day 2', 'Cage 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Cage 2', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Cage 3', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Cage 4', 'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19', 'Unnamed: 20', 'Cage 5', 'Unnamed: 22', 'Unnamed: 23', 'Unnamed: 24', 'Unnamed: 25', 'Cage 6', 'Unnamed: 27', 'Unnamed: 28']
     Day 2 Cage 1 Unnamed: 2 Unnamed: 3  Unnamed: 4 Unnamed: 5 Cage 2  \
0      NaN  match    Winners     Losers         NaN        NaN  match   
1  trial 1   2vs3        1.3        1.2         NaN    trial 1   2vs3   
2  trial 1   4vs1        1.1        1.4         NaN    trial 1   4vs1   

  Unnamed: 7 Unnamed: 8  Unnamed: 9 Unnamed: 10 Cage 3 Unnamed: 12  \
0    Winners     Losers         NaN         NaN  match     Winners   
1        2.3        2.2         NaN     trial 1   2vs3         3.3   
2        2.4        2.1         NaN     trial 1   4vs1         3.4   

  Unn

In [35]:
print(hier.dci())
print(hier.landau_h())
print(hier.davids_score())

0.4444
{'Improved_Landau_h': 0.2, 'p_value_r': 0.6166, 'p_value_l': 0.3834}
{1.1: 2.0, 1.2: -0.6667, 1.3: -0.6667, 1.4: -0.6667}


In [36]:
female_cols = {
    1: (2,3),
    2: (7,8),
    3: (12,13),
    4: (17,18),
    5: (22,23),
    6: (27,28)
}

In [37]:
male_cols = {
    2: (2,3),
    3: (7,8),
    5: (12,13),
    7: (17,18),
    8: (22,23)
}

In [38]:
def get_cage_data(df, winner_col, loser_col, start_row=1):

    cage = df.iloc[start_row:, [winner_col, loser_col]].copy()
    cage.columns = ["winner", "loser"]

    cage = cage.dropna()

    return cage

In [39]:
from HierarchiaPy import Hierarchia
import pandas as pd

file = r"C:\Users\sjs93\Downloads\CD1_food_comp_all_data_model.xlsx"

d1_f = pd.read_excel(file, sheet_name="Food_competition_D1_FEMALE")
d2_f = pd.read_excel(file, sheet_name="Food_Competition_D2_FEMALE")

results = []

for cage_num, (wcol, lcol) in female_cols.items():

    cage_d1 = get_cage_data(d1_f, wcol, lcol, start_row=1)

    # D2 has an extra header row
    cage_d2 = get_cage_data(d2_f, wcol, lcol, start_row=2)

    combined = pd.concat([cage_d1, cage_d2], ignore_index=True)

    hier = Hierarchia(combined, "winner", "loser")

    results.append({
        "cage": cage_num,
        "sex": "Female",
        "DCI": hier.dci(),
        "Landau_h": hier.landau_h()["Improved_Landau_h"]
    })

female_results = pd.DataFrame(results)

print(female_results)


   cage     sex     DCI  Landau_h
0     1  Female  0.3714       0.5
1     2  Female  0.7778       1.0
2     3  Female  0.5429       1.4
3     4  Female  0.3889       0.9
4     5  Female  0.6471       1.4
5     6  Female  0.4444       0.9


In [40]:
# Female Cage 3

cage_d1 = get_cage_data(d1_f, 12, 13, start_row=1)
cage_d2 = get_cage_data(d2_f, 12, 13, start_row=2)

combined = pd.concat([cage_d1, cage_d2], ignore_index=True)

hier = Hierarchia(combined, "winner", "loser")

print(hier.landau_h())

{'Improved_Landau_h': 1.4, 'p_value_r': 0.0929, 'p_value_l': 0.9071}


In [41]:
print("DCI:", hier.dci())
print("Landau:", hier.landau_h())
print("David's:", hier.davids_score())

DCI: 0.5429
Landau: {'Improved_Landau_h': 1.4, 'p_value_r': 0.0951, 'p_value_l': 0.9049}
David's: {3.3: 4.0, 3.1: 1.0667, 3.4: -1.0667, 3.2: -4.0}


In [42]:
print(combined)

   winner loser
0     3.3   3.2
1     3.4   3.1
2     3.3   3.4
3     3.1   3.2
4     3.3   3.1
5     3.2   3.4
6     3.3   3.1
7     3.3   3.2
8     3.4   3.2
9     3.4   3.1
10    3.2   3.1
11    3.3   3.4
12    3.1   3.2
13    3.3   3.2
14    3.4   3.3
15    3.4   3.2
16    3.1   3.3
17    3.1   3.4
18    3.3   3.2
19    3.1   3.4
20    3.2   3.4
21    3.1   3.3
22    3.3   3.4
23    3.1   3.2
24    3.3   3.4
25    3.1   3.2
26    3.3   3.2
27    3.1   3.1
28    3.4   3.2
29    3.3   3.1
30    3.1   3.4
31    3.3   3.2
32    3.4   3.2
33    3.3   3.1
34    3.3   3.4
35    3.1   3.2


In [43]:
for cage_num in female_cols.keys():

    cage_d1 = get_cage_data(d1_f, *female_cols[cage_num], start_row=1)
    cage_d2 = get_cage_data(d2_f, *female_cols[cage_num], start_row=2)

    combined = pd.concat([cage_d1, cage_d2])

    bad = combined[combined["winner"] == combined["loser"]]

    if len(bad) > 0:
        print(f"\nCage {cage_num}")
        print(bad)


Cage 1
   winner loser
19    1.1   1.1

Cage 3
   winner loser
11    3.1   3.1

Cage 5
   winner loser
17    5.1   5.1
4     5.2   5.2
